# OCSF-RainNet — Kaggle preflight (21 Nov 2026)
Audit pilot/provenance and independently execute six model forward-graph smoke tests. **NO training** and **NO held-out skill claim**: only one timestamp-consistent and not yet navigationally certified pilot scene exists. Results are diagnostic artifacts, not paper results.


In [ ]:
import os, sys, json, hashlib, platform, traceback, datetime
from pathlib import Path
import numpy as np
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")
input_root=next((p.parent for p in Path("/kaggle/input").rglob("pilot_audit_summary.json")),Path("/kaggle/input/ocsf-rainnet-private-pilot"))
work=Path("/kaggle/working"); work.mkdir(exist_ok=True)
report={"project":"OCSF-RainNet","mode":"PROVENANCE_AND_ARCHITECTURE_SMOKE_ONLY","account":"agusperdana","time_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),"scientific_gate":"BLOCKED_FOR_MODEL_TRAINING_AND_HELD_OUT_SCORES","warnings":["Single timestamp-consistent pilot pair is not independent train/validation/test data","JPEG source is rendered map and not yet independently navigated or radiometrically calibrated","No paper accuracy/Dice/IoU is computed"]}
assert input_root.is_dir(), f"Missing Kaggle private dataset: {list(Path('/kaggle/input').glob('*'))}"
with open(input_root/"pilot_audit_summary.json", encoding="utf8") as f: audit=json.load(f)
report["provenance_audit"]=audit
assert audit["currently_usable_caption_matched_pairs"]==1 and audit["currently_usable_days"]==1
d=np.load(input_root/"20240107_1400_B13_IMERG_V07.npz",allow_pickle=False)
report["available_arrays"]={k:list(d[k].shape) for k in d.files}
x=np.asarray(d["X"]); y=np.asarray(d["y"])
assert x.shape==(185,460,3) and y.shape==(185,460), (x.shape,y.shape)
count=np.bincount(y[y!=255].astype(int),minlength=5).tolist()
assert count==[77637,5806,1223,368,66], count
report["pilot"]={"shape_X":list(x.shape),"shape_y":list(y.shape),"class_counts":count,"normalized_range":[float(x.min()),float(x.max())],"no_data_pixels":int((y==255).sum())}
assert np.isfinite(x).all() and x.min()>=0 and x.max()<=1 and y.min()>=0
ckpt=input_root/"rainnet_binary_best_05.h5"
sha=hashlib.sha256(ckpt.read_bytes()).hexdigest()
expected_sha="71e0e6d7e0fa9fa6f541c89f1bfcc57d701bd94730e0508c2dbf45c341de575f"
assert sha==expected_sha, f"Original checkpoint SHA mismatch: {sha}"
report["checkpoint"]={"sha256":sha,"verified":True,"bytes":ckpt.stat().st_size}
print("SOURCE_AUDIT_PASSED",json.dumps(report["pilot"],sort_keys=True))



In [ ]:
def ordinal_numpy_unit_test():
    rng=np.random.default_rng(42)
    raw=rng.normal(size=(1,32,32,4)).astype(np.float64)
    softplus=lambda z:np.logaddexp(0,z)
    logits=np.concatenate([raw[...,:1],raw[...,:1]-np.cumsum(softplus(raw[...,1:]),axis=-1)],axis=-1)
    sigmoid=lambda z:1.0/(1.0+np.exp(-z))
    q=sigmoid(logits)
    p=np.concatenate([1-q[...,:1],q[...,:-1]-q[...,1:],q[...,-1:]],axis=-1)
    assert np.all(q[...,:-1]>=q[...,1:]-1e-13)
    assert np.all(p>=-1e-13) and np.max(abs(p.sum(axis=-1)-1.0))<1e-12
    return {"passed":True,"probability_sum_abs_max":float(np.max(abs(p.sum(axis=-1)-1.0))),"nonnegative":bool(np.all(p>=0))}
report["ordinal_numpy_smoke"]=ordinal_numpy_unit_test()
try:
    import tensorflow as tf
    report["tensorflow_version"]=tf.__version__
    report["tensorflow_gpu_devices"]=[d.name for d in tf.config.list_physical_devices("GPU")]
    sys.path.insert(0,str(input_root))
    import rainnet_intensity_models as models
    models.CHECKPOINT=ckpt
    models.ARCH=input_root
    variants=[("plain",142637),("fusion",148234),("ordinal",142632),("proposed",148217),("U-Net",485853),("RA-UNet-style",508249)]
    results=[]
    for name,count_expected in variants:
        try:
            tf.keras.backend.clear_session()
            model=(models.build_unet_baseline(residual_attention=name=="RA-UNet-style") if name in ("U-Net","RA-UNet-style") else models.build_rainnet_variant(name,pretrained=True,freeze_backbone=True))
            xi=tf.convert_to_tensor(x[:64,:64,:][None],dtype=tf.float32)
            yp=model(xi,training=False).numpy()
            assert yp.shape==(1,64,64,5)
            assert np.isfinite(yp).all() and np.min(yp)>=-1e-6
            maxerr=float(abs(yp.sum(axis=-1)-1).max())
            assert maxerr<1e-4, maxerr
            parameters=model.count_params()
            assert parameters==count_expected,(name,parameters,count_expected)
            results.append({"model":name,"status":"PASS","parameters":parameters,"shape":list(yp.shape),"sum_error_max":maxerr})
        except Exception as exc:
            results.append({"model":name,"status":"FAIL","error":repr(exc),"traceback":traceback.format_exc()[-1800:]})
    report["architecture_smoke"]=results
except Exception as exc:
    report["tensorflow_smoke_error"]=repr(exc)
    report["tensorflow_smoke_traceback"]=traceback.format_exc()[-3000:]
report["training_performed"]=False
report["held_out_results_produced"]=False
target=work/"OCSF_RainNet_Kaggle_Preflight_Report.json"
target.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding="utf-8")
print("DIAGNOSTIC_RESULT_FILE",str(target))
print("GPU",report.get("tensorflow_gpu_devices","unavailable"))
print("ARCHITECTURE_SMOKE",json.dumps(report.get("architecture_smoke",[])))
print("SCIENTIFIC_GATE",report["scientific_gate"])

